# B3 — Dominant & Significant Artworks

Identifies dominant (top-scoring) and significant artworks.

- **Dominant**: highest log-normalized dwell time per trajectory (1 per trajectory).
- **Significant**: artworks with interaction counts across all trajectories exceeding 2x the global average. These are the artworks that attract disproportionately more visitors.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('helpers'))
from fuseki_config import query_endpoint, update_endpoint
from fuseki_queries import execute_sparql_query, execute_sparql_update, format_results

In [ ]:
PREFIXES = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi:      <http://w3id.org/trajecto/poi#>
PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX xsd:      <http://www.w3.org/2001/XMLSchema#>
"""

## 1. Get All Interactions with Dwell Times

In [ ]:
query = PREFIXES + """
SELECT ?traj ?interaction ?poi ?dwell
WHERE {
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:hasInteraction ?interaction .
    ?interaction poi:interactsWith ?poi .
    ?interaction poi:hasDwellTimeInSeconds ?dwell .
    ?interaction a poi:MeaningfulInteraction .
}
"""
result = execute_sparql_query(query_endpoint, query)
df = format_results(result)
df['dwell'] = df['dwell'].astype(float)
print(f'Meaningful interactions: {len(df)}')
print(f'Trajectories: {df["traj"].nunique()}')
df.head()

## 2. Score & Rank Artworks per Trajectory (Dominant)

In [ ]:
# Score = log-normalized dwell time per trajectory
dominant_interactions = []

for traj, group in df.groupby('traj'):
    log_dwell = np.log1p(group['dwell'].values)
    max_log = log_dwell.max()
    if max_log > 0:
        scores = log_dwell / max_log
    else:
        scores = np.ones(len(group))

    group = group.copy()
    group['score'] = scores
    group = group.sort_values('score', ascending=False)

    # Top 1 = Dominant
    top = group.iloc[0]
    dominant_interactions.append({
        'traj': traj,
        'interaction': top['interaction'],
        'poi': top['poi'],
        'score': top['score'],
        'dwell': top['dwell'],
    })

print(f'Dominant interactions: {len(dominant_interactions)}')

## 3. Insert Dominant Interactions

In [ ]:
insert_lines = []
for d in dominant_interactions:
    insert_lines.append(f'  <{d["interaction"]}> a poi:DominantInteraction .')

if insert_lines:
    triples = '\n'.join(insert_lines)
    update = PREFIXES + f"""INSERT DATA {{\n{triples}\n}}"""
    success = execute_sparql_update(update_endpoint, update)
    print(f'Inserted {len(insert_lines)} DominantInteraction types: {"OK" if success else "FAILED"}')

## 4. Identify Significant POIs (Global Threshold)

A POI is significant if either:
1. Its total interaction count across all trajectories exceeds 2x the global average.
2. It is the dominant artwork for 5+ trajectories.

In [ ]:
# Criterion 1: interaction count > 2x global average
sig_query = PREFIXES + """
SELECT ?poi (COUNT(?interaction) AS ?interactionCount)
WHERE {
    ?interaction a poi:POIInteraction ;
                 poi:interactsWith ?poi .
}
GROUP BY ?poi
"""
result = execute_sparql_query(query_endpoint, sig_query)
poi_counts = format_results(result)
poi_counts['interactionCount'] = poi_counts['interactionCount'].astype(int)

avg_count = poi_counts['interactionCount'].mean()
threshold = avg_count * 2
sig_by_count = set(poi_counts[poi_counts['interactionCount'] > threshold]['poi'])

# Criterion 2: dominant artwork for 5+ trajectories
sig_by_dominant = set()
dominant_poi_counts = pd.DataFrame(dominant_interactions).groupby('poi').size()
for poi, count in dominant_poi_counts.items():
    if count >= 5:
        sig_by_dominant.add(poi)

significant_pois = sig_by_count | sig_by_dominant

print(f'Average interaction count: {avg_count:.1f}, threshold (2x): {threshold:.1f}')
print(f'Significant by interaction count: {len(sig_by_count)}')
print(f'Significant by dominant in 5+ trajectories: {len(sig_by_dominant)}')
print(f'Total significant POIs (union): {len(significant_pois)}')

# Insert
insert_lines = [f'  <{poi}> a poi:SignificantPOI .' for poi in significant_pois]

if insert_lines:
    batch_size = 500
    total = 0
    for i in range(0, len(insert_lines), batch_size):
        batch = insert_lines[i:i+batch_size]
        triples = '\n'.join(batch)
        update = PREFIXES + f"""INSERT DATA {{\n{triples}\n}}"""
        if execute_sparql_update(update_endpoint, update):
            total += len(batch)
    print(f'Inserted {total} SignificantPOI types')

## 5. Verify

In [ ]:
for label, type_uri in [('DominantInteraction', 'poi:DominantInteraction'),
                         ('SignificantPOI', 'poi:SignificantPOI')]:
    q = PREFIXES + f'SELECT (COUNT(DISTINCT ?x) AS ?c) WHERE {{ ?x a {type_uri} }}'
    result = execute_sparql_query(query_endpoint, q)
    df_v = format_results(result)
    print(f'{label}: {df_v.iloc[0]["c"] if not df_v.empty else "?"}')

# Show dominant artwork per trajectory sample
q = PREFIXES + """
SELECT ?traj ?poi ?label ?dwell
WHERE {
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:hasInteraction ?int .
    ?int a poi:DominantInteraction .
    ?int poi:interactsWith ?poi .
    ?int poi:hasDwellTimeInSeconds ?dwell .
    OPTIONAL { ?poi <http://www.w3.org/2000/01/rdf-schema#label> ?label }
}
ORDER BY ?traj
LIMIT 10
"""
result = execute_sparql_query(query_endpoint, q)
format_results(result)